In [2]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("..")

In [11]:
import json
import yaml
from pathlib import Path

import pandas as pd

from patient_simulator.data_extraction import extract_patient_profile_from_text
from patient_simulator.misc.utils import load_openrouter_api_key, create_llm_instance

In [12]:
DATA_DIR = Path("../data/aci_bench")
CSV_PATH = DATA_DIR / "combined_virtscribe_humantrans_clean.csv"
CONVERSATIONS_DIR = DATA_DIR / "conversations_raw"
PROFILES_DIR = DATA_DIR / "extracted_profiles"

df = pd.read_csv(CSV_PATH)
print(f"{len(df)} cases loaded")
df

22 cases loaded


,split,id,dataset,patient_firstname,patient_familyname,patient_gender,patient_age,doctor_name,cc,2nd_complaints,note
0,test1,VS000,virtscribe,melissa,sanchez,female,58,hughes,follow up on mitral valve repair,none,CHIEF COMPLAINT\n\nStatus post mitral valve re...
1,test2,VS001,virtscribe,jordan,roberts,male,49,NaN,high blood pressure; palpitations,none,CHIEF COMPLAINT\n\nHigh blood pressure and pal...
2,test1,VS002,virtscribe,diana,scott,female,100,NaN,heart murmur,hypolipidemia; dizziness; lightheadedness,CHIEF COMPLAINT\n\nHeart murmur.\n\nHISTORY OF...
3,test2,VS005,virtscribe,jacqueline,miller,female,40,NaN,perioral dermatitis,ocular rosacae,CHIEF COMPLAINT\n\nFollow-up for perioral derm...
4,test3,VS007,virtscribe,maria,martin,female,75,NaN,diabetes follow up,urination,CHIEF COMPLAINT\n\nFollow-up for diabetes mana...
5,test2,VS008,virtscribe,grace,ross,female,23,diaz,full spectrum STD testing,none,CHIEF COMPLAINT\n\nFull-spectrum sexually tran...
6,test2,VS009,virtscribe,amanda,taylor,female,72,NaN,hypertension,"alcohol use disorder, mild hypercholesterolemia",CHIEF COMPLAINT\n\nHypertension.\nAlcohol use ...
7,test3,VS012,virtscribe,michelle,king,female,35,phillips,acid reflux,stress,CHIEF COMPLAINT\n\nAcid reflux.\n\nHISTORY OF ...
8,test2,VS013,virtscribe,brittany,edwards,female,32,NaN,IBS,"nausea, vomiting",CHIEF COMPLAINT\n\nFollow up irritable bowel s...
9,valid,VS014,virtscribe,christina,cooper,female,65,NaN,iron deficiency anemia,chilling sensation; anxiety; depression,CHIEF COMPLAINT\n\nIron deficiency anemia.\n\n...


In [14]:
for id in df["id"].unique():
    print(f" - {id}")

 - VS000
 - VS001
 - VS002
 - VS005
 - VS007
 - VS008
 - VS009
 - VS012
 - VS013
 - VS014
 - VS015
 - VS017
 - VS018
 - VS019
 - VS022
 - VS023
 - VS024
 - VS030
 - VS035
 - VS036
 - VS037
 - VS038


## Extract patient profiles from the note column

Use `extract_patient_profile_from_text` with the `note` column (the structured clinical note, not the raw dialogue).

In [ ]:
api_key = load_openrouter_api_key(keys_file="../keys.json")

api_llm_config = """backend: APILLM
name: gemini-3.1-flash-lite-preview
vertexai: true
generation_config:
  temperature: 1
  top_p: 0.95
  max_output_tokens: 65535
  safety_settings:
    - category: HARM_CATEGORY_HATE_SPEECH
      threshold: "OFF"
    - category: HARM_CATEGORY_DANGEROUS_CONTENT
      threshold: "OFF"
    - category: HARM_CATEGORY_SEXUALLY_EXPLICIT
      threshold: "OFF"
    - category: HARM_CATEGORY_HARASSMENT
      threshold: "OFF"
  thinking_config:
    thinking_level: LOW
"""

api_config_dict = yaml.safe_load(api_llm_config)
api_llm = create_llm_instance(api_config_dict, keys_file=api_key)

In [7]:
PROFILES_DIR.mkdir(parents=True, exist_ok=True)


async def extract_and_save_profiles(df: pd.DataFrame) -> None:
    for _, row in df.iterrows():
        case_id = row["id"]
        out_path = PROFILES_DIR / f"{case_id}.json"
        if out_path.exists():
            print(f"{case_id}: skipped (already exists)")
            continue
        profile = await extract_patient_profile_from_text(
            transcript_text=row["note"],
            llm=api_llm,
        )
        out_path.write_text(json.dumps(profile, indent=4), encoding="utf-8")
        print(f"{case_id}: done")


# await extract_and_save_profiles(df)
print(f"\nProfiles saved to {PROFILES_DIR}")


Profiles saved to ../data/aci_bench/extracted_profiles


In [8]:
sample_id = df["id"].iloc[0]
profile = json.loads((PROFILES_DIR / f"{sample_id}.json").read_text())
print(f"Sample profile ({sample_id}):")
print(json.dumps(profile, indent=2))

Sample profile (VS000):
{
  "age": "58",
  "gender": "Female",
  "race": "Unknown",
  "tobacco": "Unknown",
  "alcohol": "Unknown",
  "illicit_drug": "Unknown",
  "sexual_history": "Unknown",
  "exercise": "Active within limits",
  "marital_status": "Unknown",
  "children": "Unknown",
  "living_situation": "Unknown",
  "occupation": "Unknown",
  "insurance": "Unknown",
  "allergies": "Unknown",
  "family_medical_history": "Unknown",
  "medical_device": "None reported",
  "medical_history": "Mitral regurgitation; Atrial fibrillation; Diabetes Type II; Asthma; Status post mitral valve repair 08/03/2020",
  "present_illness_positive": "Intermittent chest pain; Shallow breathing resulting in decreased exertion; Shortness of breath lasting a few minutes; Irregularly irregular cardiac rhythm; S1 slightly accentuated; Reduced breath sounds; Trace peripheral edema",
  "present_illness_negative": "No new symptoms; No JVD; No S3",
  "chiefcomplaint": "Status post mitral valve repair",
  "pain": 

# Step 2: Clean up transcripts

In [ ]:
import re
from pathlib import Path


SPEAKER_RE = re.compile(r"^(?P<prefix>[A-Za-z]+:)\s*(?P<body>.*)$")


def normalize_spacing(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?])(\S)", r"\1 \2", text)
    return text.strip()


def normalize_contractions(text: str) -> str:
    text = re.sub(r"\b(\w+)\s+n't\b", r"\1n't", text)
    text = re.sub(r"\b(\w+)\s+'(re|ve|ll|d|m|s)\b", r"\1'\2", text)
    return text


def capitalize_sentences(text: str) -> str:
    text = re.sub(r"\bi\b", "I", text)
    chars = list(text)
    capitalize_next = True
    for i, ch in enumerate(chars):
        if ch.isalpha() and capitalize_next:
            chars[i] = ch.upper()
            capitalize_next = False
        if ch in ".!?":
            capitalize_next = True
        elif ch.isalpha():
            capitalize_next = False
    text = "".join(chars)
    text = re.sub(
        r"\b(dr|mr|mrs|ms)\.",
        lambda match: f"{match.group(1).capitalize()}.",
        text,
        flags=re.IGNORECASE,
    )
    return text


def clean_line(line: str) -> str:
    stripped = line.strip()
    if not stripped:
        return ""

    match = SPEAKER_RE.match(stripped)
    if not match:
        text = normalize_spacing(stripped)
        text = normalize_contractions(text)
        text = capitalize_sentences(text)
        return text

    prefix = match.group("prefix")
    body = match.group("body")
    body = normalize_spacing(body)
    body = normalize_contractions(body)
    body = capitalize_sentences(body)
    return f"{prefix} {body}"


def clean_transcript(content: str) -> str:
    lines = content.splitlines()
    cleaned_lines = [clean_line(line) for line in lines]
    return "\n".join(cleaned_lines) + "\n"


CONVERSATIONS_CLEANED_DIR = DATA_DIR / "conversations"
CONVERSATIONS_CLEANED_DIR.mkdir(parents=True, exist_ok=True)

source_paths = sorted(CONVERSATIONS_DIR.glob("*.txt"))
if not source_paths:
    raise ValueError(f"No transcript files found in {CONVERSATIONS_DIR}")

for source_path in source_paths:
    cleaned = clean_transcript(source_path.read_text(encoding="utf-8"))
    target_path = CONVERSATIONS_CLEANED_DIR / source_path.name
    target_path.write_text(cleaned, encoding="utf-8")

print(f"Cleaned {len(source_paths)} transcripts into {CONVERSATIONS_CLEANED_DIR}")

Cleaned 22 transcripts into ../data/aci_bench/conversations_cleaned
